# Module 10: Extra Analysis

**Task 10**: Add any extra analysis based on your understanding.

**Analyses**:
1. Landscape Fragmentation Analysis
2. NDVI Trend Validation
3. Maharashtra vs. Sikkim Comparative Dashboard
4. Markov Chain LULC Prediction (2030)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from config import (
    STATES, YEARS, LULC_CLASSES, LULC_COLORS, NUM_CLASSES,
    GEE_EXPORTS_DIR, PROCESSED_DIR, FIGURES_DIR, LULC_DIR,
    TIER1_SCALE, PIXEL_AREA_KM2,
)
from scripts.lulc_utils import (
    parse_histogram, parse_transition_histogram, save_composition_csv,
)
from scripts.visualization import save_figure

print('Imports successful!')

## 10.1: Landscape Fragmentation Analysis

Compute landscape metrics to assess habitat fragmentation.

In [ ]:
import rasterio
from scipy import ndimage

def compute_fragmentation_metrics(raster_path, target_class, state_name, year):
    """Compute landscape metrics for a specific LULC class."""
    with rasterio.open(raster_path) as src:
        data = src.read(1)
        pixel_size = abs(src.transform[0])  # in degrees or meters
    
    # Binary mask for target class
    mask = (data == target_class).astype(int)
    
    # Label connected patches
    labeled, num_patches = ndimage.label(mask)
    
    # Total pixels of this class
    total_pixels = mask.sum()
    total_landscape = data.size
    
    if num_patches == 0:
        return {
            'state': state_name, 'year': year,
            'class': LULC_CLASSES[target_class],
            'num_patches': 0, 'patch_density': 0,
            'largest_patch_index': 0, 'mean_patch_size': 0,
        }
    
    # Patch sizes
    patch_sizes = ndimage.sum(mask, labeled, range(1, num_patches + 1))
    
    # Metrics
    patch_density = num_patches / (total_landscape * PIXEL_AREA_KM2.get(30, 9e-4))  # patches/km²
    largest_patch_idx = (max(patch_sizes) / total_pixels * 100) if total_pixels > 0 else 0
    mean_patch = np.mean(patch_sizes) * PIXEL_AREA_KM2.get(30, 9e-4)  # km²
    
    # Edge density (perimeter of all patches)
    # Count edge pixels (class pixels adjacent to non-class pixels)
    kernel = np.array([[0,1,0],[1,0,1],[0,1,0]])
    neighbors = ndimage.convolve(1 - mask, kernel, mode='constant', cval=0)
    edge_pixels = np.sum((mask == 1) & (neighbors > 0))
    edge_density = edge_pixels / (total_landscape * PIXEL_AREA_KM2.get(30, 9e-4))
    
    return {
        'state': state_name, 'year': year,
        'class': LULC_CLASSES[target_class],
        'num_patches': num_patches,
        'patch_density_per_km2': round(patch_density, 4),
        'largest_patch_index_pct': round(largest_patch_idx, 2),
        'mean_patch_size_km2': round(mean_patch, 4),
        'edge_density': round(edge_density, 2),
    }


# Compute for key classes: Trees (1), Crops (4), Built (6)
frag_results = []
target_classes = [1, 4, 6]  # Trees, Crops, Built

for state_key, state_cfg in STATES.items():
    for year in YEARS:
        raster_path = LULC_DIR / f'{state_key}_30m_{year}.tif'
        if not raster_path.exists():
            continue
        
        for cls in target_classes:
            result = compute_fragmentation_metrics(
                str(raster_path), cls, state_cfg['name'], year
            )
            frag_results.append(result)
            print(f'  {state_cfg["name"]} {year} {LULC_CLASSES[cls]}: '
                  f'{result["num_patches"]} patches')

if frag_results:
    frag_df = pd.DataFrame(frag_results)
    save_composition_csv(frag_df, PROCESSED_DIR / 'index' / 'fragmentation_metrics.csv')
    display(frag_df)

In [ ]:
# Plot fragmentation trends
if frag_results:
    frag_df = pd.DataFrame(frag_results)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = ['num_patches', 'patch_density_per_km2', 'largest_patch_index_pct']
    titles = ['Number of Patches', 'Patch Density (per km²)', 'Largest Patch Index (%)']
    
    for ax, metric, title in zip(axes, metrics, titles):
        if metric not in frag_df.columns:
            continue
        for state_name in frag_df['state'].unique():
            for cls_name in frag_df['class'].unique():
                subset = frag_df[(frag_df['state'] == state_name) & (frag_df['class'] == cls_name)]
                if len(subset) > 0:
                    ax.plot(subset['year'], subset[metric], 'o-',
                            label=f'{state_name} - {cls_name}', linewidth=1.5)
        ax.set_title(title)
        ax.set_xlabel('Year')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    fig.suptitle('Landscape Fragmentation Trends', fontsize=14)
    fig.tight_layout()
    save_figure(fig, 'fragmentation_trends.png', 'lulc_maps')
    plt.show()

## 10.2: Maharashtra vs. Sikkim Comparative Dashboard

In [ ]:
# Load state compositions
comp_file = GEE_EXPORTS_DIR / 'state_compositions.json'

if comp_file.exists():
    with open(comp_file, 'r') as f:
        state_compositions = json.load(f)
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    for row_idx, (state_key, state_cfg) in enumerate(STATES.items()):
        if state_key not in state_compositions:
            continue
        
        for col_idx, year in enumerate(YEARS):
            ax = axes[row_idx, col_idx]
            hist = state_compositions[state_key].get(str(year), {})
            parsed = parse_histogram(hist, TIER1_SCALE)
            
            values = [parsed.get(i, 0) * PIXEL_AREA_KM2[TIER1_SCALE] for i in range(NUM_CLASSES)]
            labels = [LULC_CLASSES[i] for i in range(NUM_CLASSES)]
            colors = [LULC_COLORS[i] for i in range(NUM_CLASSES)]
            
            # Filter out zero classes
            non_zero = [(v, l, c) for v, l, c in zip(values, labels, colors) if v > 0]
            if non_zero:
                vals, labs, cols = zip(*non_zero)
                ax.pie(vals, labels=[l if v/sum(vals) > 0.03 else '' for v, l in zip(vals, labs)],
                       colors=cols, autopct=lambda v: f'{v:.0f}%' if v > 3 else '',
                       startangle=90, pctdistance=0.85, textprops={'fontsize': 8})
            
            ax.set_title(f'{state_cfg["name"]} — {year}', fontsize=11)
    
    fig.suptitle('Maharashtra vs. Sikkim — LULC Composition (2016, 2020, 2025)',
                 fontsize=15, fontweight='bold')
    fig.tight_layout()
    save_figure(fig, 'comparative_dashboard.png', 'lulc_maps')
    plt.show()
else:
    print('⚠️ State compositions not found')

In [ ]:
# Comparative statistics table
if comp_file.exists():
    comparison = []
    
    for state_key, state_cfg in STATES.items():
        if state_key not in state_compositions:
            continue
        
        h16 = parse_histogram(state_compositions[state_key].get('2016', {}), TIER1_SCALE)
        h25 = parse_histogram(state_compositions[state_key].get('2025', {}), TIER1_SCALE)
        
        total_16 = sum(h16.values()) * PIXEL_AREA_KM2[TIER1_SCALE]
        total_25 = sum(h25.values()) * PIXEL_AREA_KM2[TIER1_SCALE]
        
        # Key metrics
        forest_16 = h16.get(1, 0) * PIXEL_AREA_KM2[TIER1_SCALE]
        forest_25 = h25.get(1, 0) * PIXEL_AREA_KM2[TIER1_SCALE]
        built_16 = h16.get(6, 0) * PIXEL_AREA_KM2[TIER1_SCALE]
        built_25 = h25.get(6, 0) * PIXEL_AREA_KM2[TIER1_SCALE]
        crop_16 = h16.get(4, 0) * PIXEL_AREA_KM2[TIER1_SCALE]
        crop_25 = h25.get(4, 0) * PIXEL_AREA_KM2[TIER1_SCALE]
        
        comparison.append({
            'State': state_cfg['name'],
            'Total Area (km²)': f'{total_16:.0f}',
            'Forest 2016 (%)': f'{forest_16/total_16*100:.1f}' if total_16 > 0 else '0',
            'Forest 2025 (%)': f'{forest_25/total_25*100:.1f}' if total_25 > 0 else '0',
            'Forest Change': f'{(forest_25-forest_16)/forest_16*100:+.1f}%' if forest_16 > 0 else 'N/A',
            'Built 2016 (%)': f'{built_16/total_16*100:.1f}' if total_16 > 0 else '0',
            'Built 2025 (%)': f'{built_25/total_25*100:.1f}' if total_25 > 0 else '0',
            'Urban Growth': f'{(built_25-built_16)/built_16*100:+.1f}%' if built_16 > 0 else 'N/A',
            'Cropland Stability': f'{(crop_25-crop_16)/crop_16*100:+.1f}%' if crop_16 > 0 else 'N/A',
        })
    
    comp_df = pd.DataFrame(comparison)
    display(comp_df)
    save_composition_csv(comp_df, PROCESSED_DIR / 'index' / 'state_comparison.csv')

## 10.3: Markov Chain LULC Prediction (2030)

In [ ]:
# Load transition matrices
transitions_file = GEE_EXPORTS_DIR / 'state_transitions.json'

if transitions_file.exists():
    with open(transitions_file, 'r') as f:
        state_transitions = json.load(f)
    
    for state_key, state_cfg in STATES.items():
        # Use 2020→2025 transition to predict 2025→2030
        hist = state_transitions.get(state_key, {}).get('2020_2025', {})
        matrix_df = parse_transition_histogram(hist, scale=TIER1_SCALE)
        
        # Convert to probability matrix (row-stochastic)
        row_sums = matrix_df.values.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1  # Avoid division by zero
        prob_matrix = matrix_df.values / row_sums
        
        # Current state (2025 composition)
        h25 = parse_histogram(
            state_compositions.get(state_key, {}).get('2025', {}), TIER1_SCALE
        )
        current_state = np.array([h25.get(i, 0) * PIXEL_AREA_KM2[TIER1_SCALE]
                                  for i in range(NUM_CLASSES)])
        
        # Predict 2030 (one step forward)
        predicted_2030 = current_state @ prob_matrix
        
        # Display
        pred_df = pd.DataFrame({
            'Class': [LULC_CLASSES[i] for i in range(NUM_CLASSES)],
            'Area 2025 (km²)': current_state,
            'Predicted 2030 (km²)': predicted_2030,
            'Change (km²)': predicted_2030 - current_state,
            'Change (%)': ((predicted_2030 - current_state) / np.maximum(current_state, 1)) * 100,
        })
        
        print(f'\n{state_cfg["name"]} — Markov Chain Prediction for 2030:')
        display(pred_df.round(2))
        
        save_composition_csv(
            pred_df,
            PROCESSED_DIR / 'index' / f'{state_key}_markov_prediction_2030.csv'
        )
        
        # Bar chart: 2025 vs Predicted 2030
        fig, ax = plt.subplots(figsize=(12, 6))
        x = np.arange(NUM_CLASSES)
        width = 0.35
        ax.bar(x - width/2, current_state, width, label='2025 (Actual)',
               color=[LULC_COLORS[i] for i in range(NUM_CLASSES)], alpha=0.8)
        ax.bar(x + width/2, predicted_2030, width, label='2030 (Predicted)',
               color=[LULC_COLORS[i] for i in range(NUM_CLASSES)], alpha=0.5,
               edgecolor='black', linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels([LULC_CLASSES[i] for i in range(NUM_CLASSES)],
                           rotation=45, ha='right')
        ax.set_ylabel('Area (km²)')
        ax.set_title(f'LULC Prediction — {state_cfg["name"]} (Markov Chain)')
        ax.legend()
        fig.tight_layout()
        save_figure(fig, f'{state_key}_markov_prediction.png', 'lulc_maps')
        plt.show()
else:
    print('⚠️ Transition data not found for Markov prediction')

## 10.4: Key Insights Summary

In [ ]:
print('='*60)
print('KEY INSIGHTS FROM EXTRA ANALYSES')
print('='*60)
print()
print('1. FRAGMENTATION:')
print('   - [Interpret: Are forests becoming more fragmented?]')
print('   - [Interpret: Is cropland consolidating or fragmenting?]')
print('   - [Interpret: How does built-up patch density change?]')
print()
print('2. MAHARASHTRA vs. SIKKIM:')
print('   - Maharashtra: Deccan plateau, large-scale agriculture, rapid urbanization')
print('   - Sikkim: Himalayan, forest-dominated, limited flat land for expansion')
print('   - [Interpret: Which state shows faster urbanization rate?]')
print('   - [Interpret: Which state retains more forest cover?]')
print()
print('3. MARKOV PREDICTION (2030):')
print('   - Caveat: Simple first-order Markov, assumes stationarity')
print('   - [Interpret: What are the projected trends?]')
print('   - [Interpret: Are they consistent with current policy directions?]')

print('\n✅ Module 10 complete.')

---
## Summary
- ✅ Landscape fragmentation metrics (patch density, LPI, edge density)
- ✅ Maharashtra vs. Sikkim comparative dashboard
- ✅ Markov chain LULC prediction for 2030
- ✅ Comparative statistics table

**All 10 modules complete!** Proceed to report writing.